Baseline Models & Feature Engineering
## Goal
Build a working time-series-safe baseline pipeline for Germany electricity price forecasting.

## Main outputs
- Feature matrix
- Missing value treatment
- Extreme value treatment decision
- Walk-forward validation framework
- Baseline 1: Naive forecast
- Baseline 2: Linear Regression / Ridge
- Baseline 3: XGBoost
- MAE / RMSE benchmark table

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

try:
    from xgboost import XGBRegressor
    xgb_available = True
except ImportError:
    xgb_available = False
    print("xgboost is not installed. The XGBoost section will be skipped.")

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

plt.rcParams["figure.figsize"] = (14, 6)

In [ ]:
%pip install xgboost


In [ ]:
df_final = pd.read_csv(r"C:\Users\wladl\Desktop\Project1\WattWise\data\df_final.csv")

print("Dataset loaded.")
print("Shape:", df_final.shape)

display(df_final.head())

In [ ]:
# 3. CREATE WORKING COPY AND PREPARE TIMESTAMP
# ============================================================

df = df_final.copy()

df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")
df = df.sort_values("timestamp").reset_index(drop=True)

print("Working dataframe created.")
print("Date range:", df["timestamp"].min(), "to", df["timestamp"].max())

display(df.head())

We convert the timestamp column to datetime format and sort the dataset in chronological order.

This is critical in time series forecasting because:
- lag features depend on time order
- rolling windows depend on time order
- train/validation/test splits must respect time order

In [ ]:
# 6. TREAT MISSING VALUES
# ============================================================

# Drop rows where target is missing
df = df.dropna(subset=["price"]).copy()

# Select numeric columns
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()

# Do not impute the target here
predictor_numeric_cols = [col for col in numeric_cols if col != "price"]

# Forward fill, then backward fill for numeric predictors
df[predictor_numeric_cols] = df[predictor_numeric_cols].ffill().bfill()

# Check remaining missing values
remaining_missing = df.isna().sum().sort_values(ascending=False)

print("Remaining missing values after treatment:")
display(remaining_missing[remaining_missing > 0].head(50))

In [ ]:
# 7. CHECK EXTREME VALUES IN PRICE
# ============================================================

print(df["price"].describe())

high_q = df["price"].quantile(0.99)
low_q = df["price"].quantile(0.01)

print("1st percentile:", low_q)
print("99th percentile:", high_q)

plt.figure(figsize=(12, 4))
plt.boxplot(df["price"].dropna(), vert=False)
plt.title("Boxplot of Price")
plt.xlabel("Price")
plt.grid(True)
plt.show()

We inspect the distribution and extreme tails of the target variable.

Decision:
- keep extreme prices
- do not cap the target

Reason:
Extreme price events are not just noise in power markets.
For a battery trading strategy, these hours are especially important.

In [ ]:
# 8. OPTIONAL: CAP EXTREME PREDICTOR VALUES ONLY
# ============================================================

def cap_outliers_iqr(series, factor=3.0):
    """
    Caps extreme values using an IQR rule.
    This is intended for predictor variables, not the target.
    """
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - factor * iqr
    upper = q3 + factor * iqr
    return series.clip(lower=lower, upper=upper)

# Example: keep this OFF by default
cap_predictors = False

if cap_predictors:
    cols_to_cap = [col for col in predictor_numeric_cols if col not in ["price_lag_24h", "price_lag_168h"]]
    for col in cols_to_cap:
        df[col] = cap_outliers_iqr(df[col])

print("Predictor capping applied:", cap_predictors)

# 9. Feature Engineering Plan

We will engineer features that capture market dynamics:

## Price memory features
- `price_lag_24`
- `price_lag_48`
- `price_lag_168`

## Rolling statistics
- `price_roll_mean_24`
- `price_roll_std_24`
- `price_roll_mean_168`
- `price_roll_std_168`

## Calendar features
- hour
- day of week
- month
- weekend flag

## Market structure features
- renewable penetration ratio
- residual load

These features reflect:
- daily repetition
- weekly repetition
- volatility
- calendar seasonality
- renewable supply pressure
- net load pressure

In [ ]:


# 10. CALENDAR FEATURES
# ============================================================

df["hour"] = df["timestamp"].dt.hour
df["day_of_week"] = df["timestamp"].dt.dayofweek
df["month"] = df["timestamp"].dt.month
df["day_of_month"] = df["timestamp"].dt.day
df["is_weekend"] = df["day_of_week"].isin([5, 6]).astype(int)

display(df[["timestamp", "hour", "day_of_week", "month", "day_of_month", "is_weekend"]].head())

These features help the model learn repeating calendar effects.

Examples:
- prices often differ by hour of day
- weekends often behave differently from weekdays
- winter and summer can have different price regimes

In [ ]:

# ============================================================
# 11. LAG FEATURES
# ============================================================

df["price_lag_24"] = df["price"].shift(24)
df["price_lag_48"] = df["price"].shift(48)
df["price_lag_168"] = df["price"].shift(168)

display(df[["timestamp", "price", "price_lag_24", "price_lag_48", "price_lag_168"]].head(200).tail(10))

Lag features bring past price information into the current row.

Meaning:
- `price_lag_24`: same hour yesterday
- `price_lag_48`: same hour two days ago
- `price_lag_168`: same hour last week

These are usually among the strongest baseline predictors in electricity markets.

In [ ]:

# ============================================================
# 12. ROLLING FEATURES
# ============================================================

df["price_roll_mean_24"] = df["price"].shift(1).rolling(window=24).mean()
df["price_roll_std_24"] = df["price"].shift(1).rolling(window=24).std()

df["price_roll_mean_168"] = df["price"].shift(1).rolling(window=168).mean()
df["price_roll_std_168"] = df["price"].shift(1).rolling(window=168).std()

display(df[[
    "timestamp",
    "price",
    "price_roll_mean_24",
    "price_roll_std_24",
    "price_roll_mean_168",
    "price_roll_std_168"
]].head(200).tail(10))

Rolling features summarize recent history.

Important detail:
- we use `.shift(1)` before the rolling window
- this avoids leakage from the current hour into its own features

These features help represent:
- recent average price level
- recent volatility
- short-term and weekly regime changes

In [ ]:

# ============================================================
# 13. RENEWABLE PENETRATION RATIO AND RESIDUAL LOAD
# ============================================================

# Forecast-based renewable total
renewable_cols = [col for col in ["wind_offshore", "wind_onshore", "solar"] if col in df.columns]

if len(renewable_cols) > 0:
    df["renewable_total"] = df[renewable_cols].sum(axis=1)
else:
    df["renewable_total"] = np.nan

# Prefer forecast load if available
if "load" in df.columns:
    df["load_used"] = df["load"]
elif "actual_load" in df.columns:
    df["load_used"] = df["actual_load"]
else:
    df["load_used"] = np.nan

# Renewable penetration ratio
df["renewable_penetration_ratio"] = np.where(
    df["load_used"] > 0,
    df["renewable_total"] / df["load_used"],
    np.nan
)

# Residual load
df["residual_load"] = df["load_used"] - df["renewable_total"]

display(df[[
    "timestamp",
    "renewable_total",
    "load_used",
    "renewable_penetration_ratio",
    "residual_load"
]].head())



## Renewable penetration ratio
This measures how much of load is covered by wind and solar.

Higher values often mean:
- more supply pressure
- lower prices
- more negative-price risk

## Residual load
This is demand that remains after renewables are subtracted.

Higher residual load often means:
- more conventional generation needed
- higher marginal cost
- upward pressure on price

In [ ]:

# ============================================================
# 14. KEEP CALENDAR COLUMNS AS NUMERIC CODES
# ============================================================

# For the first baseline models, we keep:
# hour, day_of_week, month as integer columns
# XGBoost can work with these directly
# Linear models can still use them as a simple baseline

display(df[["hour", "day_of_week", "month"]].head())

For a first baseline, we keep calendar variables as simple integer columns.



In [ ]:

# ============================================================
# 15. DROP ROWS WITH FEATURE-GENERATION NaNs
# ============================================================

before_drop = len(df)

feature_generation_cols = [
    "price_lag_24",
    "price_lag_48",
    "price_lag_168",
    "price_roll_mean_24",
    "price_roll_std_24",
    "price_roll_mean_168",
    "price_roll_std_168",
    "renewable_penetration_ratio",
    "residual_load"
]

feature_generation_cols = [col for col in feature_generation_cols if col in df.columns]

df_model = df.dropna(subset=feature_generation_cols + ["price"]).copy()

after_drop = len(df_model)

print("Rows before dropping warm-up NaNs:", before_drop)
print("Rows after dropping warm-up NaNs:", after_drop)
print("Rows removed:", before_drop - after_drop)

Lag and rolling features naturally create missing values at the start of the series.

Example:
- you cannot calculate `price_lag_168` until at least 168 hours have passed

So we drop the first warm-up rows after feature creation.

In [ ]:
#17. Finalize candidate feature list
# ============================================================
# 16. FINALIZE FEATURE LIST
# ============================================================

candidate_features = [
    # Calendar
    "hour",
    "day_of_week",
    "month",
    "day_of_month",
    "is_weekend",

    # Engineered price-memory features
    "price_lag_24",
    "price_lag_48",
    "price_lag_168",
    "price_roll_mean_24",
    "price_roll_std_24",
    "price_roll_mean_168",
    "price_roll_std_168",

    # Market structure
    "renewable_penetration_ratio",
    "residual_load",

    # Common market predictors if available
    "load",
    "wind_offshore",
    "wind_onshore",
    "solar",
    "temperature",
    "wind_speed",
    "gas_price",
    "coal_price",
    "co2_price",
]

feature_cols = [col for col in candidate_features if col in df_model.columns]

target_col = "price"

print("Number of selected features:", len(feature_cols))
print("Selected features:")
print(feature_cols)

This is the first baseline feature set.

It combines:
- calendar effects
- price memory
- rolling context
- renewable pressure
- residual load
- core market fundamentals if available

In [ ]:
# 17. FEATURE MATRIX PREVIEW
# ============================================================

X = df_model[feature_cols].copy()
y = df_model[target_col].copy()

print("X shape:", X.shape)
print("y shape:", y.shape)

display(X.head())
display(y.head())

At this point:
- `X` contains the model inputs
- `y` contains the target price

This is the feature matrix we will use for baseline modeling.

In [ ]:

# ============================================================
# 18. TIME-BASED TRAIN / VALIDATION / TEST SPLIT
# ============================================================

n = len(df_model)

train_end = int(n * 0.70)
val_end = int(n * 0.85)

train_df = df_model.iloc[:train_end].copy()
val_df = df_model.iloc[train_end:val_end].copy()
test_df = df_model.iloc[val_end:].copy()

X_train = train_df[feature_cols]
y_train = train_df[target_col]

X_val = val_df[feature_cols]
y_val = val_df[target_col]

X_test = test_df[feature_cols]
y_test = test_df[target_col]

print("Train period:", train_df["timestamp"].min(), "to", train_df["timestamp"].max(), "| rows:", len(train_df))
print("Validation period:", val_df["timestamp"].min(), "to", val_df["timestamp"].max(), "| rows:", len(val_df))
print("Test period:", test_df["timestamp"].min(), "to", test_df["timestamp"].max(), "| rows:", len(test_df))

This split respects time order:

- first 70%: training
- next 15%: validation
- last 15%: final test

Why this is correct:
- models only learn from the past
- validation simulates future unseen data
- test gives a final honest benchmark

In [ ]:

# ============================================================
# 19. VISUALIZE THE CHRONOLOGICAL SPLIT
# ============================================================

plt.figure(figsize=(16, 5))
plt.plot(df_model["timestamp"], df_model["price"], label="Price")

plt.axvline(train_df["timestamp"].max(), linestyle="--", label="Train/Val split")
plt.axvline(val_df["timestamp"].max(), linestyle="--", label="Val/Test split")

plt.title("Time-Based Train / Validation / Test Split")
plt.xlabel("Timestamp")
plt.ylabel("Price")
plt.legend()
plt.grid(True)
plt.show()

This chart confirms that the split is chronological.

That is essential in time series forecasting because future information must not leak into model training.

In [ ]:

# ============================================================
# 20. WALK-FORWARD VALIDATION FRAMEWORK
# ============================================================

def evaluate_predictions(y_true, y_pred, model_name):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = mean_squared_error(y_true, y_pred, squared=False)
    return pd.DataFrame({
        "model": [model_name],
        "MAE": [mae],
        "RMSE": [rmse]
    })

def walk_forward_splits(df_in, n_splits=3, train_ratio=0.6, val_ratio=0.2):
    """
    Creates expanding-window walk-forward splits.
    Returns a list of (train_idx, val_idx).
    """
    n = len(df_in)
    splits = []

    first_train_end = int(n * train_ratio)
    val_size = int(n * val_ratio)

    for i in range(n_splits):
        train_end = first_train_end + i * val_size
        val_end = train_end + val_size

        if val_end > n:
            break

        train_idx = np.arange(0, train_end)
        val_idx = np.arange(train_end, val_end)

        splits.append((train_idx, val_idx))

    return splits

wf_splits = walk_forward_splits(df_model, n_splits=3, train_ratio=0.5, val_ratio=0.15)

print("Number of walk-forward splits:", len(wf_splits))
for i, (tr_idx, va_idx) in enumerate(wf_splits, start=1):
    print(f"Split {i}: train={len(tr_idx)} rows, val={len(va_idx)} rows")

This creates an expanding-window validation framework.

Meaning:
- the training window grows over time
- each validation window is later in time than its training window



In [ ]:

# ============================================================
# 21. BASELINE 1A — NAIVE: YESTERDAY'S PRICE
# ============================================================

val_pred_naive_24 = X_val["price_lag_24"]
test_pred_naive_24 = X_test["price_lag_24"]

val_results_naive_24 = evaluate_predictions(y_val, val_pred_naive_24, "Naive_24h_Validation")
test_results_naive_24 = evaluate_predictions(y_test, test_pred_naive_24, "Naive_24h_Test")

display(val_results_naive_24)
display(test_results_naive_24)

This is the simplest realistic benchmark:
predict the current price as the price at the same hour yesterday.



In [ ]:

# ============================================================
# 22. BASELINE 1B — NAIVE: LAST WEEK SAME HOUR
# ============================================================

val_pred_naive_168 = X_val["price_lag_168"]
test_pred_naive_168 = X_test["price_lag_168"]

val_results_naive_168 = evaluate_predictions(y_val, val_pred_naive_168, "Naive_168h_Validation")
test_results_naive_168 = evaluate_predictions(y_test, test_pred_naive_168, "Naive_168h_Test")

display(val_results_naive_168)
display(test_results_naive_168)

This baseline predicts the price using the same hour one week earlier.



In [ ]:

# ============================================================
# 23. BASELINE 1C — HOUR-OF-WEEK AVERAGE
# ============================================================

train_df["hour_of_week"] = train_df["day_of_week"] * 24 + train_df["hour"]
val_df["hour_of_week"] = val_df["day_of_week"] * 24 + val_df["hour"]
test_df["hour_of_week"] = test_df["day_of_week"] * 24 + test_df["hour"]

hour_of_week_avg = train_df.groupby("hour_of_week")["price"].mean()

val_pred_hourweek = val_df["hour_of_week"].map(hour_of_week_avg)
test_pred_hourweek = test_df["hour_of_week"].map(hour_of_week_avg)

# fallback in case of missing mappings
global_train_mean = train_df["price"].mean()
val_pred_hourweek = val_pred_hourweek.fillna(global_train_mean)
test_pred_hourweek = test_pred_hourweek.fillna(global_train_mean)

val_results_hourweek = evaluate_predictions(y_val, val_pred_hourweek, "HourOfWeekAvg_Validation")
test_results_hourweek = evaluate_predictions(y_test, test_pred_hourweek, "HourOfWeekAvg_Test")

display(val_results_hourweek)
display(test_results_hourweek)

This baseline predicts using the average price for each hour of the week.

Example:
- Monday 08:00 has its own average
- Sunday 03:00 has its own average

This captures calendar seasonality without using advanced ML.

In [ ]:

# ============================================================
# 24. BASELINE 2 — LINEAR REGRESSION
# ============================================================

linear_model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LinearRegression())
])

linear_model.fit(X_train, y_train)

val_pred_linear = linear_model.predict(X_val)
test_pred_linear = linear_model.predict(X_test)

val_results_linear = evaluate_predictions(y_val, val_pred_linear, "LinearRegression_Validation")
test_results_linear = evaluate_predictions(y_test, test_pred_linear, "LinearRegression_Test")

display(val_results_linear)
display(test_results_linear)

Linear Regression is the first machine learning baseline.

It learns one coefficient per feature and combines them linearly.



In [ ]:

# ============================================================
# 25. BASELINE 2B — RIDGE REGRESSION
# ============================================================

ridge_model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", Ridge(alpha=1.0))
])

ridge_model.fit(X_train, y_train)

val_pred_ridge = ridge_model.predict(X_val)
test_pred_ridge = ridge_model.predict(X_test)

val_results_ridge = evaluate_predictions(y_val, val_pred_ridge, "Ridge_Validation")
test_results_ridge = evaluate_predictions(y_test, test_pred_ridge, "Ridge_Test")

display(val_results_ridge)
display(test_results_ridge)

Ridge Regression is similar to Linear Regression, but it penalizes overly large coefficients.

This often improves stability when:
- features are correlated
- the feature set is moderately large
- noisy predictors are present

In [ ]:

# ============================================================
# 26. BASELINE 3 — XGBOOST
# ============================================================

if xgb_available:
    xgb_model = XGBRegressor(
        n_estimators=300,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="reg:squarederror",
        random_state=42
    )

    xgb_model.fit(X_train, y_train)

    val_pred_xgb = xgb_model.predict(X_val)
    test_pred_xgb = xgb_model.predict(X_test)

    val_results_xgb = evaluate_predictions(y_val, val_pred_xgb, "XGBoost_Validation")
    test_results_xgb = evaluate_predictions(y_test, test_pred_xgb, "XGBoost_Test")

    display(val_results_xgb)
    display(test_results_xgb)
else:
    print("Skipping XGBoost because xgboost is not installed.")

XGBoost is a tree-based gradient boosting model.


- it handles nonlinear relationships well
- it can capture interactions automatically
- it usually performs much better than simple linear models on structured tabular data

In [ ]:

# ============================================================
# 27. COLLECT VALIDATION RESULTS
# ============================================================

validation_results = pd.concat([
    val_results_naive_24,
    val_results_naive_168,
    val_results_hourweek,
    val_results_linear,
    val_results_ridge,
] + ([val_results_xgb] if xgb_available else []), ignore_index=True)

validation_results = validation_results.sort_values("MAE").reset_index(drop=True)

display(validation_results)

This table compares all models on the validation set.



In [8]:

# ============================================================
# 28. COLLECT TEST RESULTS
# ============================================================

test_results = pd.concat([
    test_results_naive_24,
    test_results_naive_168,
    test_results_hourweek,
    test_results_linear,
    test_results_ridge,
] + ([test_results_xgb] if xgb_available else []), ignore_index=True)

test_results = test_results.sort_values("MAE").reset_index(drop=True)

display(test_results)

NameError: name 'test_results_naive_24' is not defined

This table shows final out-of-sample performance on the test set.

The test set should only be used after model choices are already made from validation.
This gives a more honest estimate of real future performance.

In [ ]:

# ============================================================
# 29. PLOT ACTUAL VS PREDICTED ON VALIDATION
# ============================================================

plt.figure(figsize=(16, 6))
plt.plot(val_df["timestamp"], y_val.values, label="Actual", linewidth=2)
plt.plot(val_df["timestamp"], val_pred_naive_24.values, label="Naive 24h", alpha=0.8)
plt.plot(val_df["timestamp"], val_pred_ridge, label="Ridge", alpha=0.8)

if xgb_available:
    plt.plot(val_df["timestamp"], val_pred_xgb, label="XGBoost", alpha=0.8)

plt.title("Validation Set: Actual vs Predicted")
plt.xlabel("Timestamp")
plt.ylabel("Price")
plt.legend()
plt.grid(True)
plt.show()

This plot helps visually compare model behavior.



In [ ]:

# ============================================================
# 30. XGBOOST FEATURE IMPORTANCE
# ============================================================

if xgb_available:
    importance_df = pd.DataFrame({
        "feature": feature_cols,
        "importance": xgb_model.feature_importances_
    }).sort_values("importance", ascending=False)

    display(importance_df.head(20))

    plt.figure(figsize=(10, 6))
    plt.barh(importance_df["feature"].head(15)[::-1], importance_df["importance"].head(15)[::-1])
    plt.title("Top 15 XGBoost Feature Importances")
    plt.xlabel("Importance")
    plt.ylabel("Feature")
    plt.grid(True)
    plt.show()
else:
    print("Feature importance skipped because xgboost is not installed.")

This gives a first view of which features matter most to the tree model.

Common strong features in power price forecasting are often:
- price_lag_24
- price_lag_168
- residual_load
- renewable penetration ratio
- gas price
- load

In [ ]:

# ============================================================
# 31. WALK-FORWARD VALIDATION EXAMPLE: RIDGE
# ============================================================

wf_results = []

for i, (train_idx, val_idx) in enumerate(wf_splits, start=1):
    train_fold = df_model.iloc[train_idx]
    val_fold = df_model.iloc[val_idx]

    X_tr = train_fold[feature_cols]
    y_tr = train_fold[target_col]

    X_va = val_fold[feature_cols]
    y_va = val_fold[target_col]

    model = Pipeline([
        ("scaler", StandardScaler()),
        ("model", Ridge(alpha=1.0))
    ])

    model.fit(X_tr, y_tr)
    pred = model.predict(X_va)

    mae = mean_absolute_error(y_va, pred)
    rmse = mean_squared_error(y_va, pred, squared=False)

    wf_results.append({
        "split": i,
        "MAE": mae,
        "RMSE": rmse,
        "train_start": train_fold["timestamp"].min(),
        "train_end": train_fold["timestamp"].max(),
        "val_start": val_fold["timestamp"].min(),
        "val_end": val_fold["timestamp"].max()
    })

wf_results_df = pd.DataFrame(wf_results)
display(wf_results_df)

This shows how a model performs across several rolling validation windows.

This is useful because one single validation split can be misleading.
Walk-forward evaluation checks whether the model is stable across time.

In [ ]:

# ============================================================
# 32. FINAL BENCHMARK SUMMARY
# ============================================================

best_val_model = validation_results.iloc[0]
best_test_model = test_results.iloc[0]

print("Best validation model:")
display(best_val_model.to_frame().T)

print("Best test model:")
display(best_test_model.to_frame().T)